
# TMDB Movie Data Analysis using Spark and PySpark

## 1. Project Introduction

This project implements a movie data analysis pipeline using Apache Spark and PySpark.

The main objectives are:
- Extract movie data from the TMDB API.
- Clean and transform movie datasets.
- Perform Exploratory Data Analysis (EDA).
- Calculate financial and popularity KPIs.
- Analyze movie franchises and directors.
- Visualize findings using Matplotlib.

## 2. Technology Stack

- Python
- Apache Spark
- PySpark
- TMDB API
- Pandas
- Matplotlib
- Jupyter Notebook



### 3. Spark Session Initialization

A SparkSession is the entry point for working with Spark DataFrames.

It allows us to:
- Read and process structured data.
- Execute Spark SQL queries.
- Perform distributed data transformations.
- Manage Spark application configurations.

The `getOrCreate()` method retrieves an existing SparkSession or creates a new one if none exists.


In [1]:

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("TMDB Movie Data Analysis") \
    .getOrCreate()

print("Spark Version:", spark.version)
print("Spark Session Created Successfully!")


d:\AmaliTech\.pyspark-env\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Spark Version: 4.3.0
Spark Session Created Successfully!



## 4. API Data Extraction

In this section, we retrieve movie data from The Movie Database (TMDB) API.

We extract:
- Movie details
- Cast information
- Crew information

The extracted data will be stored in Python dictionaries before being converted into a Spark DataFrame.



## 4. Importing the Data Extraction Function

### 4.1 Import the Extraction Module

In this section, we import the `fetch_movie_data()` function from our
custom Python module `src/extraction.py`.

We separate reusable functions from the Jupyter Notebook to improve
code organization, maintainability, and reusability.

**Why do we do this?**

- Avoid writing the same code multiple times.
- Keep the notebook focused on the data analysis workflow.
- Make our extraction logic reusable in other Python scripts.
- Follow modular programming principles.

We use Python's `sys.path` to ensure that Python can locate our
project's `src` directory when importing the function.


In [3]:

import sys
from pathlib import Path

# Get the project root directory
project_root = Path.cwd().parent

# Add project root to Python's import path
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Import the extraction function
from src.extraction import fetch_movie_data

print("Extraction function imported successfully!")


Extraction function imported successfully!



### 4.2 Understanding the Import

- `Path.cwd()` returns the current working directory of the notebook.
- `.parent` moves one directory up to the project root.
- `sys.path` contains directories Python searches for modules.
- `from src.extraction import fetch_movie_data` imports the
  function defined in `extraction.py`.

The function is now available for use in our notebook without
reimplementing its extraction logic.



## 5. API Authentication

### 5.1 Load TMDB API Credentials

To access the TMDB API, we need an API Read Access Token.

We store the token in a `.env` file located in the root directory
of our project.

We use the `python-dotenv` library to load environment variables
into our Python environment.

**Why use environment variables?**

- Protect sensitive API credentials.
- Avoid hardcoding tokens in source code.
- Prevent credentials from being exposed in Git repositories.
- Make the project easier to configure across different environments.

Our `.env` file contains:

TMDB_API_TOKEN=your_api_read_access_token


In [5]:

import os
from dotenv import load_dotenv

# Load environment variables from the project root
load_dotenv(project_root / ".env")

# Retrieve the TMDB API token
TMDB_API_TOKEN = os.getenv("TMDB_READ_ACCESS_TOKEN")

# Validate the token
if not TMDB_API_TOKEN:
    raise ValueError(
        "TMDB API token not found. Please check your .env file."
    )

# Configure API request headers
headers = {
    "Authorization": f"Bearer {TMDB_API_TOKEN}",
    "accept": "application/json"
}

print("TMDB API authentication configured successfully!")


TMDB API authentication configured successfully!



### 5.2 Understanding the Authentication Code

- `load_dotenv()` loads variables from the `.env` file.
- `os.getenv()` retrieves the API token without exposing it directly.
- The `Authorization` header sends the token to TMDB.
- The `accept` header specifies that we expect a JSON response.
- The validation ensures that the token exists before making API requests.

The token is stored in the `headers` dictionary and will be passed
to our `fetch_movie_data()` extraction function.



### 5.3 Authentication Verification

We verify that the API credentials are configured correctly
before proceeding with data extraction.

We do not print the actual API token because it is a sensitive
credential.


In [8]:

print("Token loaded:", bool(TMDB_API_TOKEN))
print("Authorization header configured:", "Authorization" in headers)


Token loaded: True
Authorization header configured: True



## 6. Defining the Movie IDs

### 6.1 Select Movies for Analysis

We use the movie IDs provided in the project requirements
to identify the movies that we want to retrieve from TMDB.

Each movie has a unique identifier in the TMDB database.

We store these identifiers in a Python list called `movie_ids`.

This list will be passed to our `fetch_movie_data()` function
to retrieve movie details and credits.

**Note:** We include all 19 IDs provided in the assignment.
Some IDs may not return valid movie records.


In [9]:

movie_ids = [
    0, 299534, 19995, 140607, 299536, 597, 135397,
    420818, 24428, 168259, 99861, 284054, 12445,
    181808, 330457, 351286, 109445, 321612, 260513
]

print("Total movie IDs:", len(movie_ids))


Total movie IDs: 19



### 6.2 Validate the Movie ID List

Before extracting data, we perform a simple validation
to confirm that our list contains the expected number of IDs.

This helps ensure that we are using the correct input
for our extraction process.


In [10]:

assert len(movie_ids) == 19, "Expected exactly 19 movie IDs."

print("Movie ID validation successful.")


Movie ID validation successful.



## 7. Extracting Movie Data from TMDB

### 7.1 Execute the Data Extraction Function

We now execute the `fetch_movie_data()` function imported
from our `src/extraction.py` module.

The function iterates through the movie IDs and sends
HTTP GET requests to the TMDB API.

For each movie, we retrieve:
- Movie details, including title, budget, revenue, genres,
  release date, runtime, and popularity.
- Cast information, including actors and their character names.
- Crew information, including directors and other production members.

We use the `append_to_response=credits` parameter to retrieve
movie details and credits within a single API request.

The returned JSON responses are stored in the Python list
`movie_data`.

**Output:** A list of dictionaries containing the raw movie data
retrieved from TMDB.


In [11]:

movie_data = fetch_movie_data(movie_ids, headers)


Failed to fetch movie ID 0: 404 Client Error: Not Found for url: https://api.themoviedb.org/3/movie/0?append_to_response=credits&language=en-US
Fetched: 299534 - Avengers: Endgame
Fetched: 19995 - Avatar
Fetched: 140607 - Star Wars: The Force Awakens
Fetched: 299536 - Avengers: Infinity War
Fetched: 597 - Titanic
Fetched: 135397 - Jurassic World
Fetched: 420818 - The Lion King
Fetched: 24428 - The Avengers
Fetched: 168259 - Furious 7
Fetched: 99861 - Avengers: Age of Ultron
Fetched: 284054 - Black Panther
Fetched: 12445 - Harry Potter and the Deathly Hallows: Part 2
Fetched: 181808 - Star Wars: The Last Jedi
Fetched: 330457 - Frozen II
Fetched: 351286 - Jurassic World: Fallen Kingdom
Fetched: 109445 - Frozen
Fetched: 321612 - Beauty and the Beast
Fetched: 260513 - Incredibles 2

Extraction Summary
------------------
Successfully fetched: 18
Failed: 1



### 7.2 Evaluate the Extraction Results

After extraction, we evaluate the results to determine
how many movie records were successfully retrieved.

We compare the number of extracted records with the
number of requested movie IDs.

We also identify any failed requests that need further
investigation.

This validation helps us assess the completeness of
our extracted dataset before moving to data inspection
and transformation.


In [12]:

print("Requested movie IDs:", len(movie_ids))
print("Successfully extracted movies:", len(movie_data))
print("Extraction completeness:", 
      round(len(movie_data) / len(movie_ids) * 100, 2), "%")


Requested movie IDs: 19
Successfully extracted movies: 18
Extraction completeness: 94.74 %



### 7.3 Inspect the Raw Extracted Data

We inspect one movie record to understand the structure
of the JSON response returned by TMDB.

The API response contains several nested objects and arrays.

Understanding this structure is important because we will
later extract and transform these fields into a structured
Spark DataFrame.

We begin by inspecting the available keys in the first
successfully extracted movie record.


In [13]:

movie_data[0].keys()


dict_keys(['adult', 'backdrop_path', 'belongs_to_collection', 'budget', 'genres', 'homepage', 'id', 'imdb_id', 'origin_country', 'original_language', 'original_title', 'overview', 'popularity', 'poster_path', 'production_companies', 'production_countries', 'release_date', 'revenue', 'runtime', 'softcore', 'spoken_languages', 'status', 'tagline', 'title', 'video', 'vote_average', 'vote_count', 'credits'])


### 7.4 Inspect Movie Credits

The `credits` field contains information about the movie's
cast and crew.

We inspect its structure to understand how actor and director
information is organized.

The cast will later be used for actor-based movie searches,
while the crew data will be used to identify directors.


In [14]:

movie_data[0]["credits"].keys()


dict_keys(['cast', 'crew'])


### 7.5 Inspect the Raw JSON Response

We display the complete JSON response of the first movie
to explore the nested data structures.

This inspection helps us identify:
- Fields that require extraction.
- Nested JSON objects.
- Arrays containing multiple values.
- Fields that may contain missing information.

We use this information to design our data cleaning
and transformation functions.


In [15]:

import json

print(json.dumps(movie_data[0], indent=2))


{
  "adult": false,
  "backdrop_path": "/7RyHsO4yDXtBv1zUU3mTpHeQ0d5.jpg",
  "belongs_to_collection": {
    "id": 86311,
    "name": "The Avengers Collection",
    "poster_path": "/yFSIUVTCvgYrpalUktulvk3Gi5Y.jpg",
    "backdrop_path": "/2UNUv4NJdC36E5myDHACBJ99EwL.jpg"
  },
  "budget": 356000000,
  "genres": [
    {
      "id": 12,
      "name": "Adventure"
    },
    {
      "id": 878,
      "name": "Science Fiction"
    },
    {
      "id": 28,
      "name": "Action"
    }
  ],
  "homepage": "https://www.marvel.com/movies/avengers-endgame",
  "id": 299534,
  "imdb_id": "tt4154796",
  "origin_country": [
    "US"
  ],
  "original_language": "en",
  "original_title": "Avengers: Endgame",
  "overview": "After the devastating events of Avengers: Infinity War, the universe is in ruins due to the efforts of the Mad Titan, Thanos. With the help of remaining allies, the Avengers must assemble once more in order to undo Thanos' actions and restore order to the universe once and for all, no m


## 8. Saving Raw Data — Bronze Layer

### 8.1 Preserve the Raw API Responses

Before performing data cleaning and transformation, we save
the raw movie data retrieved from the TMDB API.

We store the extracted records in JSON format inside the
`data/raw/` directory.

This represents the Bronze Layer of our data pipeline.

The raw data remains unchanged so that we can reuse it
for future processing, debugging, and data recovery.

**Important:** We do not perform cleaning or remove any
fields from the raw dataset at this stage.


In [16]:


import json

# Create the raw data directory if it does not exist
raw_data_path = project_root / "data" / "raw"
raw_data_path.mkdir(parents=True, exist_ok=True)

# Define the output file
raw_file = raw_data_path / "tmdb_movies_raw.json"

# Save raw movie data
with open(raw_file, "w", encoding="utf-8") as file:
    json.dump(movie_data, file, indent=4, ensure_ascii=False)

print(f"Raw data saved successfully: {raw_file}")
print(f"Total records saved: {len(movie_data)}")



Raw data saved successfully: d:\AmaliTech\tmdb-spark-analysis\data\raw\tmdb_movies_raw.json
Total records saved: 18



### 8.2 Verify the Raw Data File

After saving the extracted data, we verify that the file
exists and contains the expected number of records.

This validation ensures that our Bronze Layer has been
successfully created before proceeding to data cleaning.


In [17]:


# Verify the saved raw dataset
with open(raw_file, "r", encoding="utf-8") as file:
    raw_data_check = json.load(file)

print("File exists:", raw_file.exists())
print("Records in saved file:", len(raw_data_check))



File exists: True
Records in saved file: 18
